# 06 - Agents

Demonstrate AIMU's agent support: autonomous agents that loop over tool calls, sequential pipelines that chain agents, and streaming output.

Per Anthropic's taxonomy:
- **Agents** (`SimpleAgent`, `SkillAgent`): the LLM autonomously directs tool use and decides when to stop
- **Workflows** (`Chain`, `Router`, `Parallel`, `EvaluatorOptimizer`): code controls the flow; LLMs are invoked at defined points

This notebook covers `SimpleAgent`, `Chain`, `AgenticModelClient`, and `SkillAgent`. See **notebook 08** for `Router`, `Parallel`, and `EvaluatorOptimizer`.

## Setup

Create an `OllamaClient` with a tool-capable model and attach an `MCPClient` with a few demo tools. The same setup is used across all sections.

In [ ]:
import datetime
from fastmcp import FastMCP

from aimu.models.ollama import OllamaClient
from aimu.tools import MCPClient

mcp = FastMCP("Demo Tools")


@mcp.tool()
def get_current_date_and_time() -> str:
    """Returns the current date and time."""
    return str(datetime.datetime.now())


@mcp.tool()
def get_weather(location: str) -> str:
    """Returns the current weather for a given location."""
    # Stubbed for demo purposes
    return f"Sunny, 22Â°C in {location}"


@mcp.tool()
def calculate(expression: str) -> str:
    """Evaluates a simple arithmetic expression and returns the result."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
model_client.mcp_client = MCPClient(server=mcp)

print("Model:", model_client.model.value)
print("Tools:", [t.name for t in model_client.mcp_client.list_tools()])

## A - Basic SimpleAgent

A `SimpleAgent` wraps a `ModelClient` and runs an agentic loop: after each `chat()` call, if the model used tools, the agent sends a continuation prompt to allow further tool use. The loop stops once the model responds without calling any tools.

This is the simplest possible usage â€” two lines.

In [ ]:
from aimu.agents import SimpleAgent

model_client.messages = []

agent = SimpleAgent(model_client, name="assistant", max_iterations=5)
result = agent.run("Get the current date and time. Subsequently calculate 123 * 456.")

print(result)

The agent called both tools â€” date/time and calculator â€” and assembled the results into a final answer. Inspect the full message history to see every step.

In [ ]:
model_client.messages

## B - Agent with System Message

Pass a `system_message` to give the agent a specific persona or set of instructions.

In [ ]:
model_client.messages = []

agent = SimpleAgent(
    model_client,
    name="weather-bot",
    max_iterations=5,
)
model_client.system_message = "You are a travel assistant. Always check the weather before giving advice. Be concise. If you've answered the original query and don't need to use any more tools, repeat the final answer."

result = agent.run("Should I visit Paris or London today?")
print(result)

In [ ]:
model_client.messages

## C - SimpleAgent from Config

`SimpleAgent.from_config()` creates an agent from a plain dict. This makes it easy to define agents in configuration files or environment settings without writing boilerplate code.

In [ ]:
agent_config = {
    "name": "calculator-agent",
    "system_message": "You are a maths assistant. Use the calculate tool for all arithmetic. If you've answered the original query and don't need to use any more tools, repeat the final answer.",
    "max_iterations": 8,
    "continuation_prompt": "Continue solving the problem step by step.",
}

model_client.messages = []
agent = SimpleAgent.from_config(agent_config, model_client)

print(f"Agent name:       {agent.name}")
print(f"Max iterations:   {agent.max_iterations}")
print(f"System message:   {model_client.system_message}")

In [ ]:
result = agent.run("What is (17 * 23) + (88 / 4)?")
print(result)

In [ ]:
model_client.messages

## D - Streaming Agent Output

`run_streamed()` yields `AgentChunk` objects â€” each with `agent_name`, `iteration`, `phase`, and `content`. This mirrors the `StreamChunk` API from `ModelClient.chat_streamed()`, extended with agent context.

The `iteration` field shows which loop round produced each chunk, making it easy to follow multi-step reasoning.

In [ ]:
from aimu.models import StreamPhase

model_client.messages = []
model_client.system_message = None

agent = SimpleAgent(model_client, name="streamer", max_iterations=6)

current_iteration = -1
current_phase = None

for chunk in agent.run_streamed("What is the weather in Tokyo, and what is 99 * 11?"):
    if chunk.iteration != current_iteration:
        current_iteration = chunk.iteration
        print(f"\n--- iteration {current_iteration} ---")

    if chunk.phase != current_phase:
        current_phase = chunk.phase
        print(f"\n--- phase {current_phase.value} ---")

    if chunk.phase == StreamPhase.THINKING:
        print(f"{chunk.content}", end="", flush=True)
    elif chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"{chunk.content['name']} > {chunk.content['response']}")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

In [ ]:
model_client.messages

## E - Multi-Step Chain

A `Chain` chains agents sequentially; the text output of each step becomes the input to the next. Each agent runs its own agentic loop independently. This is Anthropic's **Prompt Chaining** workflow pattern; AIMU names it `Chain` because the units being chained are full `Agent` objects, not raw prompts.

`Chain.from_config()` accepts a list of config dicts and a single `ModelClient`. The client is shared across all steps; before each step, `messages` are cleared and `system_message` is applied from the step's config, keeping steps isolated.

Here a three-step pipeline: a **researcher** gathers facts using tools, an **analyst** interprets them, and a **formatter** produces a clean summary.

In [ ]:
from aimu.agents import Chain

chain_configs = [
    {
        "name": "researcher",
        "system_message": (
            "You are a research assistant. Use available tools to collect facts. "
            "Output only the raw facts you found, one per line."
        ),
        "max_iterations": 6,
    },
    {
        "name": "analyst",
        "system_message": "You are a data analyst. Interpret the provided facts and draw conclusions. Be concise.",
        "max_iterations": 3,
    },
    {
        "name": "formatter",
        "system_message": (
            "You are a copywriter. Rewrite the provided analysis as a short, friendly "
            "paragraph suitable for a general audience. No bullet points."
        ),
        "max_iterations": 1,
    },
]

chain = Chain.from_config(chain_configs, model_client)

result = chain.run("Gather the current date/time and the weather in London, then summarise.")
print(result)

## F - Streaming Chain Output

`Chain.run_streamed()` yields `ChainChunk` objects that extend `AgentChunk` with a `step` index, letting you track progress across the full pipeline in real time.

In [ ]:
chain2 = AgentChain.from_config(chain_configs, model_client)

current_step = -1

for chunk in chain2.run_streamed("Get the current time and weather in Sydney, then summarise."):
    if chunk.step != current_step:
        current_step = chunk.step
        agent_name = chain_configs[current_step]["name"]
        print(f"\n=== Step {current_step}: {agent_name} ===")

    if chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"  [tool: {chunk.content['name']}] → {chunk.content['response']}")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## G - AgenticModelClient

`AgenticModelClient` wraps a `SimpleAgent` behind the standard `ModelClient` interface. Use it anywhere a plain `ModelClient` is accepted (web UIs, conversation managers, etc.) to get the full agentic loop transparently.

`chat()` runs the complete `SimpleAgent` loop and returns once the model stops calling tools. `generate()` bypasses the loop and calls the inner client directly. State (`messages`, `system_message`, `mcp_client`) delegates to `SimpleAgent.model_client`, so the wrapper is a true drop-in replacement.

To run a workflow (`AgentChain`, `Router`, etc.), call its `run()` or `run_streamed()` directly; see **notebook 08** for workflow patterns.

In [ ]:
from aimu.agents import AgenticModelClient

client = OllamaClient(OllamaClient.MODELS.QWEN_3_8B)
client.mcp_client = MCPClient(server=mcp)

agentic_client = AgenticModelClient(SimpleAgent(client, max_iterations=5))

# chat() drives the full agentic loop â€” identical call site to a plain ModelClient
result = agentic_client.chat("What is the weather in Berlin, and what is 42 * 7?")
print(result)

## H - SkillAgent

 extends  with skill support. It auto-discovers  files
from the filesystem, injects a skill catalog into the system message, and attaches an MCP
server that exposes . The model calls  to pull in full
instructions for a relevant skill before answering.

The only difference from  is the extra  parameter; every other
constructor argument,  key, and streaming API is identical.

See **notebook 07** for a deep-dive on skill discovery, the MCP server, script tools, and
more usage patterns.

In [ ]:
from aimu import paths
from aimu.agents import SkillAgent
from aimu.skills import SkillManager

skill_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
skill_client.system_message = "You are a helpful assistant. Activate relevant skills before answering."

# SkillManager points at the sample skills bundled with AIMU
skill_agent = SkillAgent(
    skill_client,
    name="skilled-assistant",
    max_iterations=6,
    skill_manager=SkillManager(skill_dirs=[str(paths.skills)]),
)

result = skill_agent.run("Write me a haiku about autumn rain.")
print(result)

## I - Clean Up

Delete model clients to release any held resources.

In [ ]:
del model_client, chain, chain2, agentic_client, skill_client